# **Pertemuan 3: Aplikasi Interaktif: Kode QR, Landmark, dan Image Overlay**

**Tujuan Sesi:**  
Peserta mengintegrasikan semua pengetahuan untuk membangun aplikasi interaktif. Konsep overlay akan diperkenalkan secara *"just-in-time"* sebagai bagian dari proyek akhir.


## **Pengantar**

### **Konsep Konvolusi, Deteksi Tepi, dan Pembacaan Kode QR**

Dalam pengolahan citra digital, **konvolusi** merupakan operasi penting yang memungkinkan kita untuk memanipulasi fitur visual dari gambar.  
Beberapa penerapannya antara lain:  
- **Blurring (reduksi noise)**: menggunakan kernel untuk meratakan nilai piksel di sekitarnya, sehingga detail halus dan noise berkurang.  
- **Sharpening**: meningkatkan kontras antar piksel agar tepi objek terlihat lebih tajam.  

Selain itu, kita juga akan mengenal **algoritma Canny Edge Detector**, salah satu metode paling populer untuk **menemukan garis batas objek** pada gambar.  
Metode ini bekerja dengan mendeteksi perubahan intensitas yang signifikan, lalu menandai area tersebut sebagai **tepi**.

### **Menulis Skrip untuk Membaca Kode QR**

Kode QR adalah representasi data yang sering digunakan untuk menyimpan informasi seperti URL, teks, atau data lainnya.  
Pada sesi ini, kita akan melakukan **hands-on** untuk:  
1. **Memuat gambar yang mengandung Kode QR**.  
2. Menggunakan **pyzbar** atau **cv2.QRCodeDetector** untuk mendeteksi dan membaca isi Kode QR.  
3. Menampilkan hasil pembacaan langsung di atas gambar menggunakan **cv2.putText**.

*Langkah-langkah ini akan membantu kita memahami bagaimana memproses gambar untuk membaca informasi tersembunyi di dalamnya.*

## **Aplikasi 1: Deteksi dan Pembacaan Kode QR**

### **Teori: Anatomi Dasar Kode QR dan Cara Kerjanya**

**Kode QR (Quick Response)** adalah jenis barcode 2D yang dapat menyimpan berbagai jenis data seperti:
- URL website
- Teks biasa
- Informasi kontak
- Koordinat GPS

**Struktur Kode QR:**
1. **Finder Patterns**: Tiga kotak besar di sudut yang membantu scanner mengenali orientasi QR
2. **Timing Patterns**: Garis hitam-putih bergantian untuk menentukan koordinat modul
3. **Alignment Patterns**: Kotak kecil untuk koreksi distorsi
4. **Format Information**: Area yang berisi informasi tentang error correction level
5. **Data Area**: Area yang berisi data aktual yang dikodekan

**Proses Deteksi QR Code:**
1. **Preprocessing**: Konversi ke grayscale dan threshold
2. **Finder Pattern Detection**: Mencari pola 1:1:3:1:1 (hitam:putih:hitam:putih:hitam)
3. **Perspective Correction**: Mengoreksi sudut pandang
4. **Decoding**: Membaca data dari area data

### **Demo: Mendeteksi Kode QR dan Menampilkan Data**

Berikut adalah demo lengkap untuk mendeteksi QR code, menggambar poligon di sekelilingnya, dan menampilkan data yang terkandung:

In [ ]:
# Import library yang dibutuhkan
import numpy as np  # Untuk operasi numerik dan manipulasi array
import matplotlib.pyplot as plt  # Untuk visualisasi gambar
import cv2  # OpenCV: library utama untuk pengolahan citra dan aplikasi terkait
import os  # Untuk operasi file dan path
import dlib  # Deteksi wajah & landmark
import pyzbar  # Pembaca barcode / QR code

# Cek versi library yang diinstal
print(f"Versi numpy: {np.__version__}")
print(f"Versi matplotlib: {plt.matplotlib.__version__}")
print(f"Versi OpenCV: {cv2.__version__}")
print("Versi dlib:", dlib.__version__)
print("Versi pyzbar:", pyzbar.__file__)


In [ ]:
def detect_and_decode_qr(image_path):
    """
    Fungsi untuk mendeteksi dan membaca QR code dari gambar
    """
    # Baca gambar
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Tidak dapat membaca gambar {image_path}")
        return None
    
    # Konversi ke RGB untuk matplotlib
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Deteksi QR codes menggunakan pyzbar
    qr_codes = pyzbar.decode(image)
    
    # Proses setiap QR code yang ditemukan
    for qr_code in qr_codes:
        # Ekstrak data
        qr_data = qr_code.data.decode('utf-8')
        qr_type = qr_code.type
        
        # Dapatkan koordinat bounding box
        points = qr_code.polygon
        
        # Jika polygon tidak ditemukan, gunakan rect
        if len(points) != 4:
            points = qr_code.rect
            x, y, w, h = points
            points = [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]
        
        # Gambar poligon di sekitar QR code
        points = np.array(points, dtype=np.int32)
        cv2.polylines(image_rgb, [points], True, (0, 255, 0), 3)
        
        # Tambahkan teks dengan data QR code
        rect = qr_code.rect
        x, y, w, h = rect
        
        # Tampilkan tipe QR dan data
        text_lines = [
            f"Type: {qr_type}",
            f"Data: {qr_data[:50]}..." if len(qr_data) > 50 else f"Data: {qr_data}"
        ]
        
        # Posisikan teks di atas QR code
        text_y = y - 10
        for i, line in enumerate(text_lines):
            text_position = (x, text_y - (i * 25))
            cv2.putText(image_rgb, line, text_position, 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
        print(f"QR Code ditemukan:")
        print(f"  Type: {qr_type}")
        print(f"  Data: {qr_data}")
        print(f"  Position: {points}")
        print("-" * 50)
    
    return image_rgb, len(qr_codes)

# Test fungsi dengan gambar sample
# result_image, qr_count = detect_and_decode_qr('sample_qr.jpg')
# if result_image is not None:
#     plt.figure(figsize=(10, 8))
#     plt.imshow(result_image)
#     plt.title(f'QR Code Detection - {qr_count} QR code(s) found')
#     plt.axis('off')
#     plt.show()

## **Praktik 1: Membuat Pembaca Kode QR**

### **Hand-on: Menulis Skrip untuk Memuat Gambar QR Code**

Mari kita buat skrip lengkap untuk membaca QR code step by step:

In [ ]:
# Step 1: Install required libraries
# pip install opencv-python pyzbar pillow matplotlib

import cv2
import numpy as np
from pyzbar import pyzbar
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# Fungsi untuk membuat QR code sample (opsional)
def create_sample_qr():
    """Membuat QR code sample untuk testing"""
    import qrcode
    
    # Data yang akan dikodekan
    data = "https://www.python.org"
    
    # Buat QR code
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    qr.add_data(data)
    qr.make(fit=True)
    
    # Buat gambar
    img = qr.make_image(fill_color="black", back_color="white")
    img.save("sample_qr.png")
    print(f"Sample QR code created with data: {data}")

# Uncomment untuk membuat sample QR code
# create_sample_qr()

In [ ]:
# Step 2: Implementasi menggunakan cv2.QRCodeDetector (alternatif)

def detect_qr_opencv(image_path):
    """
    Menggunakan OpenCV QRCodeDetector untuk deteksi QR code
    """
    # Baca gambar
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Tidak dapat membaca gambar {image_path}")
        return None
    
    # Konversi ke RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Inisialisasi QR Code detector
    qr_detector = cv2.QRCodeDetector()
    
    # Deteksi dan decode QR code
    data, points, _ = qr_detector.detectAndDecode(image)
    
    if data:
        print(f"QR Code ditemukan: {data}")
        
        # Gambar polygon jika points ditemukan
        if points is not None:
            points = points.astype(int)
            cv2.polylines(image_rgb, [points], True, (0, 255, 0), 3)
            
            # Tambahkan teks
            x, y = points[0][0]
            cv2.putText(image_rgb, f"Data: {data}", (x, y-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    else:
        print("Tidak ada QR code yang ditemukan")
    
    return image_rgb, data

# Test dengan kedua metode
# result_pyzbar, count = detect_and_decode_qr('sample_qr.png')
# result_opencv, data = detect_qr_opencv('sample_qr.png')

In [ ]:
# Step 3: Implementasi lengkap dengan error handling dan multiple QR codes

class QRCodeReader:
    def __init__(self):
        self.opencv_detector = cv2.QRCodeDetector()
    
    def read_from_image(self, image_path, method='pyzbar'):
        """
        Membaca QR code dari gambar
        method: 'pyzbar' atau 'opencv'
        """
        try:
            image = cv2.imread(image_path)
            if image is None:
                raise ValueError(f"Tidak dapat membaca gambar: {image_path}")
            
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            if method == 'pyzbar':
                return self._decode_with_pyzbar(image_rgb)
            elif method == 'opencv':
                return self._decode_with_opencv(image_rgb)
            else:
                raise ValueError("Method harus 'pyzbar' atau 'opencv'")
                
        except Exception as e:
            print(f"Error: {e}")
            return None, []
    
    def _decode_with_pyzbar(self, image):
        """Decode menggunakan pyzbar"""
        qr_codes = pyzbar.decode(image)
        results = []
        
        for qr in qr_codes:
            result = {
                'data': qr.data.decode('utf-8'),
                'type': qr.type,
                'polygon': qr.polygon,
                'rect': qr.rect
            }
            results.append(result)
            
            # Gambar polygon
            points = np.array(qr.polygon, dtype=np.int32)
            cv2.polylines(image, [points], True, (0, 255, 0), 3)
            
            # Tambahkan teks
            x, y, w, h = qr.rect
            cv2.putText(image, f"{qr.data.decode('utf-8')[:30]}...", 
                       (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        
        return image, results
    
    def _decode_with_opencv(self, image):
        """Decode menggunakan OpenCV"""
        data, points, _ = self.opencv_detector.detectAndDecode(image)
        results = []
        
        if data:
            result = {
                'data': data,
                'type': 'QRCODE',
                'polygon': points[0] if points is not None else None,
                'rect': None
            }
            results.append(result)
            
            if points is not None:
                points = points.astype(int)
                cv2.polylines(image, [points], True, (0, 255, 0), 3)
                
                x, y = points[0][0]
                cv2.putText(image, f"{data[:30]}...", (x, y-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        
        return image, results

# Contoh penggunaan
qr_reader = QRCodeReader()

# Test dengan pyzbar
# image_result, qr_results = qr_reader.read_from_image('sample_qr.png', 'pyzbar')
# print(f"Ditemukan {len(qr_results)} QR code(s)")
# for i, result in enumerate(qr_results):
#     print(f"QR {i+1}: {result['data']}")

## **Aplikasi 2: Deteksi Landmark & Image Overlay**

### **Teori: Face Landmark Detection dengan 68 Titik Kunci**

**Face Landmark** adalah titik-titik kunci pada wajah yang dapat dideteksi secara otomatis. Model standar menggunakan **68 titik** yang merepresentasikan:

1. **Jaw line (0-16)**: Garis rahang dari telinga kiri ke telinga kanan
2. **Right eyebrow (17-21)**: Alis mata kanan
3. **Left eyebrow (22-26)**: Alis mata kiri
4. **Nose bridge (27-30)**: Batang hidung
5. **Lower nose (31-35)**: Bagian bawah hidung
6. **Right eye (36-41)**: Mata kanan
7. **Left eye (42-47)**: Mata kiri
8. **Outer lip (48-59)**: Bibir luar
9. **Inner lip (60-67)**: Bibir dalam

**Library Dlib** adalah library C++ dengan Python binding yang sangat populer untuk:
- Face detection
- Facial landmark detection
- Face recognition
- Object tracking

### **Teori "Just-in-Time": Image Overlay dengan Alpha Blending**

**Alpha Blending** adalah teknik untuk menggabungkan dua gambar dengan transparansi. Konsep kunci:

**1. RGBA Color Model:**
- **R**: Red channel (0-255)
- **G**: Green channel (0-255)
- **B**: Blue channel (0-255)
- **A**: Alpha channel (0-255), di mana:
  - 0 = Transparan penuh
  - 255 = Opaque penuh

**2. Formula Alpha Blending:**
```
result_color = (alpha * foreground) + ((1 - alpha) * background)
```

**3. Implementasi untuk Image Overlay:**
- Gambar PNG dengan channel alpha sebagai mask transparansi
- Posisikan overlay berdasarkan landmark detection
- Blending dengan background menggunakan alpha values

In [ ]:
# Demo: Face Landmark Detection dan Glasses Overlay

import cv2
import dlib
import numpy as np
import matplotlib.pyplot as plt

class FaceLandmarkDetector:
    def __init__(self, predictor_path=None):
        """
        Inisialisasi detector
        predictor_path: path ke file shape_predictor_68_face_landmarks.dat
        """
        self.face_detector = dlib.get_frontal_face_detector()
        
        if predictor_path:
            self.landmark_predictor = dlib.shape_predictor(predictor_path)
        else:
            print("Warning: Predictor path tidak diberikan")
            print("Download dari: http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2")
            self.landmark_predictor = None
    
    def detect_landmarks(self, image):
        """Deteksi landmark pada gambar"""
        if self.landmark_predictor is None:
            return [], []
        
        # Konversi ke grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        
        # Deteksi wajah
        faces = self.face_detector(gray)
        
        landmarks_list = []
        face_rects = []
        
        for face in faces:
            # Deteksi landmark
            landmarks = self.landmark_predictor(gray, face)
            
            # Konversi ke array numpy
            points = []
            for i in range(68):
                x = landmarks.part(i).x
                y = landmarks.part(i).y
                points.append((x, y))
            
            landmarks_list.append(np.array(points))
            face_rects.append(face)
        
        return landmarks_list, face_rects
    
    def draw_landmarks(self, image, landmarks_list):
        """Gambar landmark pada image"""
        result = image.copy()
        
        for landmarks in landmarks_list:
            for i, (x, y) in enumerate(landmarks):
                cv2.circle(result, (int(x), int(y)), 2, (0, 255, 0), -1)
                # Tambahkan nomor titik (opsional)
                # cv2.putText(result, str(i), (int(x), int(y)), 
                #            cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 255), 1)
        
        return result

# Inisialisasi detector
# detector = FaceLandmarkDetector('shape_predictor_68_face_landmarks.dat')

In [ ]:
# Implementasi Image Overlay dengan Alpha Blending

class ImageOverlay:
    @staticmethod
    def load_overlay_image(image_path):
        """Load gambar overlay dengan alpha channel"""
        # Baca gambar dengan alpha channel
        overlay = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
        
        if overlay is None:
            raise ValueError(f"Tidak dapat membaca gambar: {image_path}")
        
        # Jika tidak ada alpha channel, buat default
        if overlay.shape[2] == 3:
            alpha = np.ones((overlay.shape[0], overlay.shape[1], 1), dtype=overlay.dtype) * 255
            overlay = np.concatenate([overlay, alpha], axis=2)
        
        return overlay
    
    @staticmethod
    def resize_overlay(overlay, target_width, target_height):
        """Resize overlay ke ukuran target"""
        return cv2.resize(overlay, (target_width, target_height))
    
    @staticmethod
    def apply_overlay(background, overlay, x, y):
        """
        Terapkan overlay pada background dengan alpha blending
        x, y: posisi top-left overlay
        """
        h_overlay, w_overlay = overlay.shape[:2]
        h_bg, w_bg = background.shape[:2]
        
        # Pastikan overlay tidak keluar dari background
        if x + w_overlay > w_bg:
            w_overlay = w_bg - x
        if y + h_overlay > h_bg:
            h_overlay = y_bg - y
        if x < 0 atau y < 0:
            return background
        
        # Crop overlay jika perlu
        overlay_crop = overlay[:h_overlay, :w_overlay]
        
        # Ekstrak alpha channel (normalize ke 0-1)
        alpha = overlay_crop[:, :, 3] / 255.0
        alpha_3ch = np.stack([alpha, alpha, alpha], axis=2)
        
        # Ekstrak RGB channels
        overlay_rgb = overlay_crop[:, :, :3]
        
        # Area background yang akan di-overlay
        bg_area = background[y:y+h_overlay, x:x+w_overlay]
        
        # Alpha blending
        blended = (alpha_3ch * overlay_rgb + (1 - alpha_3ch) * bg_area).astype(np.uint8)
        
        # Terapkan hasil blending ke background
        result = background.copy()
        result[y:y+h_overlay, x:x+w_overlay] = blended
        
        return result

# Contoh penggunaan overlay
overlay_processor = ImageOverlay()

In [ ]:
# Demo lengkap: Deteksi wajah dan penempatan kacamata

class GlassesFilter:
    def __init__(self, predictor_path, glasses_path):
        self.landmark_detector = FaceLandmarkDetector(predictor_path)
        self.overlay_processor = ImageOverlay()
        self.glasses_image = self.overlay_processor.load_overlay_image(glasses_path)
    
    def calculate_glasses_position(self, landmarks):
        """
        Hitung posisi dan ukuran kacamata berdasarkan landmark mata
        landmarks: array 68 titik landmark
        """
        # Titik mata kanan (36-41) dan mata kiri (42-47)
        right_eye = landmarks[36:42]
        left_eye = landmarks[42:48]
        
        # Hitung center dan ukuran mata
        right_eye_center = np.mean(right_eye, axis=0).astype(int)
        left_eye_center = np.mean(left_eye, axis=0).astype(int)
        
        # Hitung jarak antar mata
        eye_distance = np.linalg.norm(left_eye_center - right_eye_center)
        
        # Ukuran kacamata proporsional dengan jarak mata
        glasses_width = int(eye_distance * 2.5)
        glasses_height = int(glasses_width * 0.4)  # Aspect ratio kacamata
        
        # Posisi kacamata (center antara kedua mata)
        glasses_center_x = int((right_eye_center[0] + left_eye_center[0]) / 2)
        glasses_center_y = int((right_eye_center[1] + left_eye_center[1]) / 2)
        
        # Posisi top-left kacamata
        glasses_x = glasses_center_x - glasses_width // 2
        glasses_y = glasses_center_y - glasses_height // 2
        
        return glasses_x, glasses_y, glasses_width, glasses_height
    
    def apply_glasses_filter(self, image):
        """Terapkan filter kacamata pada image"""
        # Deteksi landmark
        landmarks_list, _ = self.landmark_detector.detect_landmarks(image)
        
        result = image.copy()
        
        # Terapkan kacamata untuk setiap wajah
        for landmarks in landmarks_list:
            # Hitung posisi kacamata
            x, y, w, h = self.calculate_glasses_position(landmarks)
            
            # Resize kacamata
            glasses_resized = self.overlay_processor.resize_overlay(self.glasses_image, w, h)
            
            # Terapkan overlay
            result = self.overlay_processor.apply_overlay(result, glasses_resized, x, y)
        
        return result, len(landmarks_list)

# Contoh penggunaan
# glasses_filter = GlassesFilter('shape_predictor_68_face_landmarks.dat', 'glasses.png')
# result_image, face_count = glasses_filter.apply_glasses_filter(input_image)
# print(f"Filter diterapkan pada {face_count} wajah")

## **Praktik 2: Proyek Mini - Filter Wajah Sederhana**

### **Goal: Membuat Filter yang Menempatkan Aksesori pada Wajah**

Mari kita buat proyek mini untuk mengimplementasikan filter wajah sederhana dengan langkah-langkah berikut:

**Langkah-langkah:**
1. **Deteksi wajah** menggunakan Haar Cascade atau Dlib
2. **Dapatkan landmark** menggunakan Dlib 68-point detector
3. **Hitung posisi dan ukuran aksesori** berdasarkan posisi landmark
4. **Terapkan Alpha Blending** untuk menempelkan gambar PNG aksesori

In [ ]:
# Proyek Mini: Face Filter Application

import cv2
import dlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

class FaceFilterApp:
    def __init__(self, predictor_path=None):
        """
        Inisialisasi aplikasi face filter
        predictor_path: path ke shape_predictor_68_face_landmarks.dat
        """
        # Inisialisasi detectors
        self.init_face_detectors(predictor_path)
        
        # Aksesori yang tersedia
        self.accessories = {}
        
        print("Face Filter App initialized!")
        if predictor_path is None:
            print("⚠️  Landmark predictor tidak tersedia")
            print("   Download dari: http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2")
    
    def init_face_detectors(self, predictor_path):
        """Inisialisasi face detectors"""
        # Haar Cascade detector (backup)
        self.haar_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
        
        # Dlib detector
        self.dlib_detector = dlib.get_frontal_face_detector()
        
        # Landmark predictor
        if predictor_path and os.path.exists(predictor_path):
            self.landmark_predictor = dlib.shape_predictor(predictor_path)
            self.has_landmarks = True
        else:
            self.landmark_predictor = None
            self.has_landmarks = False
    
    def detect_faces_haar(self, image):
        """Deteksi wajah menggunakan Haar Cascade"""
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        faces = self.haar_cascade.detectMultiScale(gray, 1.3, 5)
        return faces
    
    def detect_faces_dlib(self, image):
        """Deteksi wajah menggunakan Dlib"""
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        faces = self.dlib_detector(gray)
        return faces
    
    def get_landmarks(self, image, face_rect):
        """Dapatkan 68 landmark points"""
        if not self.has_landmarks:
            return None
        
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        landmarks = self.landmark_predictor(gray, face_rect)
        
        # Konversi ke numpy array
        points = []
        for i in range(68):
            x = landmarks.part(i).x
            y = landmarks.part(i).y
            points.append((x, y))
        
        return np.array(points)
    
    def load_accessory(self, name, image_path):
        """Load aksesori dari file"""
        try:
            accessory = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
            if accessory is None:
                raise ValueError(f"Tidak dapat membaca {image_path}")
            
            # Pastikan ada alpha channel
            if accessory.shape[2] == 3:
                alpha = np.ones((accessory.shape[0], accessory.shape[1], 1), dtype=accessory.dtype) * 255
                accessory = np.concatenate([accessory, alpha], axis=2)
            
            self.accessories[name] = accessory
            print(f"✅ Aksesori '{name}' berhasil dimuat")
            
        except Exception as e:
            print(f"❌ Error memuat aksesori '{name}': {e}")
    
    def create_simple_glasses(self):
        """Buat kacamata sederhana secara programmatis"""
        # Buat canvas transparan
        glasses = np.zeros((100, 300, 4), dtype=np.uint8)
        
        # Gambar frame kacamata
        # Lensa kiri
        cv2.circle(glasses, (75, 50), 35, (0, 0, 0, 255), 3)
        # Lensa kanan
        cv2.circle(glasses, (225, 50), 35, (0, 0, 0, 255), 3)
        # Bridge
        cv2.line(glasses, (110, 50), (190, 50), (0, 0, 0, 255), 3)
        # Temple kiri
        cv2.line(glasses, (40, 50), (10, 45), (0, 0, 0, 255), 3)
        # Temple kanan
        cv2.line(glasses, (260, 50), (290, 45), (0, 0, 0, 255), 3)
        
        self.accessories['glasses'] = glasses
        print("✅ Kacamata sederhana dibuat")
    
    def create_simple_mustache(self):
        """Buat kumis sederhana secara programmatis"""
        mustache = np.zeros((60, 120, 4), dtype=np.uint8)
        
        # Gambar kumis
        points = np.array([[10, 40], [30, 20], [60, 30], [90, 20], [110, 40], 
                          [90, 50], [60, 45], [30, 50]], np.int32)
        cv2.fillPoly(mustache, [points], (0, 0, 0, 255))
        
        self.accessories['mustache'] = mustache
        print("✅ Kumis sederhana dibuat")
    
    def calculate_accessory_position(self, landmarks, accessory_type):
        """Hitung posisi aksesori berdasarkan landmark"""
        if landmarks is None:
            return None, None, None, None
        
        if accessory_type == 'glasses':
            # Posisi berdasarkan mata (titik 36-47)
            left_eye = landmarks[42:48]
            right_eye = landmarks[36:42]
            
            eye_center = np.mean(np.concatenate([left_eye, right_eye]), axis=0)
            eye_width = np.linalg.norm(np.mean(left_eye, axis=0) - np.mean(right_eye, axis=0))
            
            width = int(eye_width * 2.5)
            height = int(width * 0.33)
            x = int(eye_center[0] - width // 2)
            y = int(eye_center[1] - height // 2)
            
        elif accessory_type == 'mustache':
            # Posisi berdasarkan hidung (titik 31-35)
            nose_bottom = landmarks[31:36]
            nose_center = np.mean(nose_bottom, axis=0)
            
            width = int(np.linalg.norm(landmarks[31] - landmarks[35]) * 2)
            height = int(width * 0.5)
            x = int(nose_center[0] - width // 2)
            y = int(nose_center[1] + height // 4)
            
        else:
            return None, None, None, None
        
        return x, y, width, height
    
    def apply_accessory(self, image, accessory, x, y, width, height):
        """Terapkan aksesori dengan alpha blending"""
        # Resize aksesori
        accessory_resized = cv2.resize(accessory, (width, height))
        
        h_acc, w_acc = accessory_resized.shape[:2]
        h_img, w_img = image.shape[:2]
        
        # Boundary checking
        if x < 0 atau y < 0 atau x + w_acc > w_img atau y + h_acc > h_img:
            return image
        
        # Alpha blending
        alpha = accessory_resized[:, :, 3] / 255.0
        alpha_3ch = np.stack([alpha, alpha, alpha], axis=2)
        
        accessory_rgb = accessory_resized[:, :, :3]
        background_area = image[y:y+h_acc, x:x+w_acc]
        
        blended = (alpha_3ch * accessory_rgb + (1 - alpha_3ch) * background_area).astype(np.uint8)
        
        result = image.copy()
        result[y:y+h_acc, x:x+w_acc] = blended
        
        return result
    
    def apply_filter(self, image, accessories_to_apply=['glasses']):
        """Terapkan filter pada gambar"""
        result = image.copy()
        faces_detected = 0
        
        # Deteksi wajah dengan Dlib
        faces = self.detect_faces_dlib(image)
        
        for face in faces:
            faces_detected += 1
            
            # Dapatkan landmarks jika tersedia
            landmarks = None
            if self.has_landmarks:
                landmarks = self.get_landmarks(image, face)
            
            # Terapkan setiap aksesori
            for accessory_name in accessories_to_apply:
                if accessory_name in self.accessories:
                    accessory = self.accessories[accessory_name]
                    
                    # Hitung posisi
                    x, y, w, h = self.calculate_accessory_position(landmarks, accessory_name)
                    
                    if x is not None:
                        result = self.apply_accessory(result, accessory, x, y, w, h)
        
        return result, faces_detected

# Inisialisasi aplikasi
import os
app = FaceFilterApp()

# Buat aksesori sederhana
app.create_simple_glasses()
app.create_simple_mustache()

print("\n🎭 Face Filter App siap digunakan!")
print(f"Aksesori tersedia: {list(app.accessories.keys())}")

In [ ]:
# Test dan demonstrasi aplikasi

def demo_face_filter():
    """Demo penggunaan face filter"""
    # Buat gambar test atau gunakan webcam
    print("Demo Face Filter Application")
    print("=" * 40)
    
    # Opsi 1: Test dengan gambar
    def test_with_image(image_path):
        try:
            image = cv2.imread(image_path)
            if image is None:
                print(f"❌ Tidak dapat membaca gambar: {image_path}")
                return
            
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Terapkan filter
            result, face_count = app.apply_filter(image_rgb, ['glasses', 'mustache'])
            
            # Tampilkan hasil
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            
            axes[0].imshow(image_rgb)
            axes[0].set_title('Original')
            axes[0].axis('off')
            
            axes[1].imshow(result)
            axes[1].set_title(f'With Filter ({face_count} faces detected)')
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"✅ Filter berhasil diterapkan pada {face_count} wajah")
            
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # Opsi 2: Test dengan webcam (real-time)
    def test_with_webcam():
        print("\n📷 Membuka webcam...")
        print("Tekan 'q' untuk keluar, 'g' untuk toggle kacamata, 'm' untuk toggle kumis")
        
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("❌ Tidak dapat membuka webcam")
            return
        
        show_glasses = True
        show_mustache = True
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Konversi ke RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Tentukan aksesori yang akan diterapkan
            accessories = []
            if show_glasses:
                accessories.append('glasses')
            if show_mustache:
                accessories.append('mustache')
            
            # Terapkan filter
            result, face_count = app.apply_filter(frame_rgb, accessories)
            
            # Konversi kembali ke BGR untuk OpenCV
            result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
            
            # Tampilkan info
            cv2.putText(result_bgr, f'Faces: {face_count}', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(result_bgr, 'Press q:quit, g:glasses, m:mustache', (10, 60), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            cv2.imshow('Face Filter Demo', result_bgr)
            
            # Handle keyboard input
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('g'):
                show_glasses = not show_glasses
                print(f"Glasses: {'ON' if show_glasses else 'OFF'}")
            elif key == ord('m'):
                show_mustache = not show_mustache
                print(f"Mustache: {'ON' if show_mustache else 'OFF'}")
        
        cap.release()
        cv2.destroyAllWindows()
        print("✅ Demo webcam selesai")
    
    # Menu pilihan
    print("Pilih mode demo:")
    print("1. Test dengan gambar (ganti 'test_image.jpg' dengan path gambar Anda)")
    print("2. Test dengan webcam (real-time)")
    print("3. Skip demo")
    
    choice = input("Pilihan (1/2/3): ").strip()
    
    if choice == '1':
        image_path = input("Masukkan path gambar: ").strip()
        if not image_path:
            image_path = 'test_image.jpg'
        test_with_image(image_path)
    elif choice == '2':
        test_with_webcam()
    else:
        print("Demo dilewati")

# Jalankan demo (uncomment untuk menjalankan)
# demo_face_filter()

## **Ringkasan dan Latihan Tambahan**

### **Apa yang Telah Dipelajari:**

1. **QR Code Detection & Reading:**
   - Anatomi dan cara kerja QR Code
   - Implementasi dengan `pyzbar` dan `cv2.QRCodeDetector`
   - Menggambar poligon dan menampilkan data

2. **Face Landmark Detection:**
   - 68 titik landmark pada wajah
   - Penggunaan library Dlib
   - Deteksi mata, hidung, mulut, dan kontur wajah

3. **Image Overlay dengan Alpha Blending:**
   - Konsep RGBA dan alpha channel
   - Formula alpha blending
   - Implementasi overlay transparan

4. **Face Filter Application:**
   - Integrasi semua komponen
   - Real-time processing dengan webcam
   - Penempatan aksesori berdasarkan landmark

### **Latihan Tambahan:**

1. **Extend QR Code Reader:**
   - Tambahkan support untuk multiple QR codes
   - Implementasikan QR code generator
   - Buat GUI untuk QR code scanner

2. **Advanced Face Filters:**
   - Buat filter topi berdasarkan landmark dahi
   - Implementasikan filter anting-anting pada telinga
   - Tambahkan animasi pada aksesori

3. **Performance Optimization:**
   - Optimasi untuk real-time processing
   - Multi-threading untuk better performance
   - Caching untuk landmark detection

### **Resources untuk Pengembangan Lebih Lanjut:**

- **Dlib Models:** http://dlib.net/files/
- **OpenCV Tutorials:** https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html
- **Face Recognition:** https://github.com/ageitgey/face_recognition
- **MediaPipe:** https://mediapipe.dev/ (Google's ML solutions)